# **Module 3.2: Residual RL with MPC** 🧠🏎️

---

**What this notebook does:** In Lab 8 we built a **Model Predictive Controller (MPC)** that
tracks a reference path by solving a small optimization problem 10 times a second. It is
safe and predictable, but it's only as good as its model: a simple kinematic bicycle that knows
nothing about tyre slip, load transfer or the actuator lag of the real car. So it is
**conservative**, and it leaves lap time on the table.

Here we keep the MPC exactly as it is and train a policy to add a small **residual
correction** on top of every MPC command:

$$u \;=\; u_{\text{MPC}} \;+\; u_{\text{RL}}$$

The MPC does most of the work (it already drives a clean lap), and the policy just learns
*where and by how much to correct it*. Since training starts from a controller that already
works, the car is safe from the first episode and needs far fewer samples than learning to
drive from scratch.

The PPO is the same from-scratch version as in the earlier notebooks (network, GAE, clipped
loss). The only things that change are what the action means and the reward.

<img src="https://raw.githubusercontent.com/dhyeyshah28/RL_for_Roboracer/main/assets/images/mpcrl.png" width="1000">


---

---

## 📝 Student TODOs — Complete These Before Running Training

This notebook has **6 TODOs** to complete. TODOs 5 and 6 are the PPO pieces from
Learn to Race; paste in your own working versions if you have them. Everything else is given to you.

| # | TODO | Location | What You Build |
|---|---|---|---|
| **TODO 1** | Speed profile for the MPC waypoints | Section 3, `speed_profile` | What the MPC tracks |
| **TODO 2** | Residual observation: MPC command and tracking errors | Section 5, `_process_obs` | What the policy sees |
| **TODO 3** | Add the residual to the MPC command | Section 5, `_apply_residual` | The residual action |
| **TODO 4** | Residual reward | Section 5, wrapper `step()` | Reward design |
| **TODO 5** | Actor-critic forward pass | Section 6, `ActorCritic.forward` | Same as Learn to Drive/Race |
| **TODO 6** | Compute GAE advantages | Section 6.2, `compute_gae` | Same as Learn to Drive/Race |

> **📌 Tip:** Read each section's explanation carefully before attempting its TODO. The
> math and logic you need is explained right above the corresponding code cell.

---

## 📋 How This Works

| | Learn the Racing Line | Residual RL on Top of MPC |
|---|---|---|
| **Who drives** | Pure pursuit, chasing a point the policy chose | The **MPC** from Lab 8, tracking the waypoints |
| **Policy outputs** | Offset from the centerline and a target speed | A small **correction** to the MPC's steering and speed |
| **Decision rate** | 20 Hz | 10 Hz, the MPC's own rate (`DTK = 0.1` s) |
| **Untrained policy** | Spins out in the first tight corner | **Is the MPC**: its average correction is zero |
| **How we judge it** | Lap time and the path on the map | Lap time **compared with the MPC alone** |






**In code:** one call to `ResidualMPCWrapper.step(action)` is one MPC decision.
1. The MPC has already solved for the current state: `u_MPC = [steer, speed]`.
2. `_apply_residual` adds the policy's correction `[Δsteer, Δspeed]` and clips to the car's limits.
3. The command is held for 10 sim steps (0.1 s).
4. The wrapper adds up the progress, computes the reward, solves the MPC again for the new
   state, and returns the new observation (which includes that next MPC command).


<img src="https://raw.githubusercontent.com/dhyeyshah28/RL_for_Roboracer/main/assets/images/residuals.png" width="700">



---

---

# **Section 1 — Concepts** 💡

## 1.1 Quick MPC Recap (Lab 8)

At every control step, MPC looks $T = 8$ steps ahead ($0.8$ s) and solves a quadratic program:

$$\min_{u_{0..T-1}} \;\sum_{t} \underbrace{(z_t - z^{\text{ref}}_t)^\top Q\,(z_t - z^{\text{ref}}_t)}_{\text{track the reference}} \;+\; \underbrace{u_t^\top R\, u_t}_{\text{small inputs}} \;+\; \underbrace{\Delta u_t^\top R_d\, \Delta u_t}_{\text{smooth inputs}}$$

$$\text{subject to}\quad z_{t+1} = A z_t + B u_t + C \;\;\text{(linearized kinematic bicycle)}, \quad \text{speed, steering and acceleration limits}$$

- State $z = [x, y, v, \psi]$, input $u = [a, \delta]$ (acceleration and steering)
- Only the **first** input is applied, then the problem is solved again from the new state (receding horizon)

## 1.2 Why we add on top of plain MPC

| Limitation | What it costs |
|---|---|
| **Model mismatch:** a kinematic bicycle, no tyre slip or load transfer | Tracking errors in fast corners, so the speed profile must stay cautious |
| **Linearization** around the predicted path, re-solved at 10 Hz | Late reactions when the car is far from the prediction |
| **Hand-tuned costs** $Q, R, R_d$ and a fixed speed profile | One compromise for the whole track |

All three are *systematic*: they show up in the same places every lap, which is exactly the
kind of thing a learned correction can pick up.

## 1.3 Residual Learning

Instead of replacing the controller, we only learn the **difference** between what the MPC
does and what would be faster:

$$u \;=\; u_{\text{MPC}}(z) \;+\; u_{\text{RL}}(o), \qquad \lvert \Delta\delta \rvert \le 0.1 \text{ rad}, \;\; \lvert \Delta v \rvert \le 2 \text{ m/s}$$

| Why residual | |
|---|---|
| **Safe from step one** | The MPC keeps the car on track while the policy explores small corrections |
| **Sample efficient** | The policy does not have to learn to drive, only to improve a driver |
| **Interpretable** | We can plot exactly where and how much the policy overrides the MPC |
| **Easy fallback** | Set the residual to zero and you are back to the plain MPC |

## 1.4 Our MDP

**State** (114 numbers, all scaled to $[0, 1]$): the 108 lidar beams, speed and steering angle
(the **same 110 values** as every earlier notebook), plus the MPC's current command
(steering and speed) and the car's **tracking errors** (sideways distance $e_y$ and heading
difference $e_\psi$ to the waypoints).

- **Why the MPC command?** A correction only makes sense if the policy knows what it is
  correcting: "+0.5 m/s" means something different when the MPC asks for 2 m/s than for 6 m/s.
- **Why the tracking errors?** The MPC already tries to drive them to zero, but only inside its
  own simple model, and it trades them off against smooth inputs. Whatever error is *left* is
  where the model is wrong (for example the car sliding wide in a fast corner), and that is
  exactly where a correction helps. The policy isn't rewarded for reducing them; it uses them
  as a measurement.
- **Are they required?** No. In principle the policy could work this out from the lidar, but
  that is much harder to learn. Giving it the base controller's command and error signals is
  the usual choice in residual RL.

**Action:** two numbers $a_0, a_1 \in [-1, 1]$, turned into a correction:

$$\Delta\delta \;=\; a_0 \cdot 0.1 \text{ rad} \qquad\qquad \Delta v \;=\; a_1 \cdot 2.0 \text{ m/s}$$

**Reward** for one decision, written the same way as in the earlier notebooks:

$$r_t = \Delta_{\text{progress}} - 0.5\max(0,\ 0.4 - d_{\min}) - 0.05\,(a_0^2 + a_1^2) - 10\cdot\mathbb{1}[\text{collision or spin-out}]$$

- **Progress** $\Delta_{\text{progress}}$: metres gained along the centerline this decision (the same progress signal as before)
- **Wall penalty** (same as Learn to Race): $d_{\min}$ is the closest lidar distance; it costs a little whenever the car is closer than 0.4 m to a wall
- **Residual penalty** (new): a small cost on the size of the correction, so the policy only overrides the MPC where it pays off.
  - $a_0, a_1$ are the raw policy outputs in $[-1, 1]$, before scaling. Steering and speed are in different units, so penalizing the raw outputs makes "half the allowed correction" cost the same on both
  - Squaring makes small nudges almost free ($0.1^2 = 0.01$) and big overrides expensive ($1^2 = 1$)
  - With 0.05, even a full correction on both channels costs 0.1, about 20% of the ~0.5 m of progress per decision, so a correction only survives where it actually gains time
- **Crash penalty** (same as Learn the Racing Line): hitting a wall or spinning out ends the episode

Episodes last a fixed 30 s, so the most progress is the same as the **fastest lap**.


<img src="https://raw.githubusercontent.com/dhyeyshah28/RL_for_Roboracer/main/assets/images/residualscar.png" width="550">



## 1.5 Keeping It Physically Real

Same rule as Learn the Racing Line: the simulator's tyre model never runs out of grip, so the
sideways acceleration $a_{\text{lat}} = v\,\omega$ (filtered over about 0.2 s) is tracked, and going past
$1.1\,\mu g \approx 11.3$ m/s² is a **spin-out**, treated exactly like a crash. So the policy
can't buy a faster lap by asking the tyres for more grip than a real car has.

---

---

# **Section 2 — Setup** 🔧

In [ ]:
# ── Install all dependencies ──────────────────────────────────────────────────
!pip install -q git+https://github.com/f1tenth/f1tenth_gym.git@dev-humble
!pip install -q stable-baselines3[extra]
!pip install -q wandb imageio imageio-ffmpeg cvxpy

print("Installation complete!")

## 2.2 Verify Imports

In [ ]:
# ── Verify everything imports correctly ───────────────────────────────────────
import os
import copy
import math
import time
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)   # scipy block_diag notice, printed on every MPC solve
warnings.filterwarnings("ignore", category=FutureWarning)
from dataclasses import dataclass, field
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
from collections import deque
import cvxpy
from scipy.sparse import block_diag, csc_matrix
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.collections import LineCollection
from IPython.display import clear_output, Image, display
import wandb
import f1tenth_gym

# SB3 is only used for the vectorized envs; PPO itself is written from scratch below
from stable_baselines3.common.vec_env import (
    SubprocVecEnv, DummyVecEnv, VecMonitor, VecNormalize
)

print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"cvxpy solvers:  {cvxpy.installed_solvers()}")
print("Imports OK")

## 2.3 Meet the Track

Load Spielberg's map and centerline, plus the plotting helpers we use for every plot in this
notebook (including zoomed views of the three sharpest corners).

In [ ]:
# ── Load the track + plotting helpers ─────────────────────────────────────────
from f1tenth_gym.envs.track import Track

MAP_NAME = "Spielberg"
TRACK    = Track.from_track_name(MAP_NAME)      # also caches the map files
CL       = TRACK.centerline
CL_XY    = np.column_stack([CL.xs, CL.ys])
TRACK_LENGTH = float(CL.length)
HALF_WIDTH   = 1.1                              # metres, from Spielberg_centerline.csv

_occ  = np.array(TRACK.occupancy_map)
_spec = TRACK.spec
_res  = _spec.resolution
_ox, _oy = _spec.origin[0], _spec.origin[1]
_EXTENT  = [_ox, _ox + _occ.shape[1] * _res, _oy, _oy + _occ.shape[0] * _res]

# The 3 sharpest corners (largest |curvature|, at least 25 m apart), used for zoom views
_ss, _ks = np.asarray(CL.ss), np.asarray(CL.ks)
CORNER_S = []
for i in np.argsort(-np.abs(_ks)):
    if all(min(abs(_ss[i] - s), TRACK_LENGTH - abs(_ss[i] - s)) > 25 for s in CORNER_S):
        CORNER_S.append(float(_ss[i]))
    if len(CORNER_S) == 3:
        break
CORNER_XY = [TRACK.frenet_to_cartesian(s, 0.0, 0.0)[:2] for s in CORNER_S]

V_PLOT_MIN, V_PLOT_MAX = 0.0, 8.0               # colour scale for speed (m/s)
CL_COLOR, MPC_COLOR, RL_COLOR = "#777777", "#00bcd4", "#ff9800"


def track_ax(ax, zoom_xy=None, zoom=7.0):
    """Draw the occupancy map on ax; optionally zoom to a box around zoom_xy."""
    ax.imshow(_occ, cmap="gray", origin="lower", extent=_EXTENT, vmin=0, vmax=255)
    if zoom_xy is None:
        ax.set_xlim(CL_XY[:, 0].min() - 3, CL_XY[:, 0].max() + 3)
        ax.set_ylim(CL_XY[:, 1].min() - 3, CL_XY[:, 1].max() + 3)
    else:
        ax.set_xlim(zoom_xy[0] - zoom, zoom_xy[0] + zoom)
        ax.set_ylim(zoom_xy[1] - zoom, zoom_xy[1] + zoom)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])


def speed_line(ax, path, lw=2.5):
    """Draw an (N, >=3) [x, y, speed, ...] path on ax, coloured by speed."""
    pts  = path[:, :2].reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = LineCollection(segs, cmap="turbo", norm=plt.Normalize(V_PLOT_MIN, V_PLOT_MAX), linewidth=lw)
    lc.set_array(path[:-1, 2])
    ax.add_collection(lc)
    return lc


def plain_line(ax, xy, color, label=None, lw=1.4, ls="-"):
    ax.plot(xy[:, 0], xy[:, 1], color=color, lw=lw, ls=ls, label=label)


plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "legend.fontsize": 10})
print(f"{MAP_NAME}: {TRACK_LENGTH:.0f} m, sharpest corners at s = {[round(s) for s in CORNER_S]} m")

---

# **Section 3 — The Waypoints the MPC Tracks** 🧭

The MPC follows a list of waypoints `[x, y, yaw, speed]` (in Lab 8 these came from
`waypoint_logger.py` as `waypoints.csv`). We use the **centerline**, which keeps the most
margin to the walls, and give it a **speed profile** from the same grip physics as Learn the
Racing Line:

$$v_{\text{corner}}(s) \;=\; \min\!\Big(v_{\max},\; \sqrt{\frac{a_{\text{lat}}}{\lvert\kappa(s)\rvert}}\Big) \qquad\qquad v(s) \;\le\; \sqrt{v(s + \Delta s)^2 + 2\,a_{\text{brake}}\,\Delta s}$$

- The first rule: the fastest speed each point's curvature allows, using a cautious $a_{\text{lat}} = 4$ m/s²
- The second rule (a **backward pass** around the closed lap): the car must be able to brake down to the next point's speed in time

This profile is deliberately **safe**. The MPC tracks it comfortably, and it leaves room for
the residual policy to find places where the car can go faster.

In [ ]:
# ── Waypoints: centerline + speed profile ─────────────────────────────────────
PROFILE_A_LAT  = 4.0    # m/s^2, cautious cornering grip for the profile
PROFILE_A_BRK  = 4.0    # m/s^2, braking the profile assumes
PROFILE_V_MAX  = 6.0    # m/s, same as the MPC's MAX_SPEED


def speed_profile(ks, ds, a_lat=PROFILE_A_LAT, a_brake=PROFILE_A_BRK, v_max=PROFILE_V_MAX):
    """Speed (m/s) at every waypoint: corner limit from the curvature, then a
    backward braking pass so the car can always slow down in time."""

    # ═════════════════════════════════════════════════════════════════
    # ✏️  TODO 1 — Speed Profile for the MPC Waypoints
    # ═════════════════════════════════════════════════════════════════

    # Start from the corner-speed limit at every waypoint (capped at v_max).
    # Then walk backwards around the lap, lowering each speed so the car can
    # brake (at a_brake over the distance ds[i]) to the next waypoint's speed.
    # Do the backward pass twice, because the lap is a closed loop.
    # Return: v (one speed per waypoint)

    # ─────────────────────────────────────────────────────────────────





    # ═════════════════════════════════════════════════════════════════
    pass  # remove once you've implemented the steps above

_ds = np.diff(np.append(_ss, TRACK_LENGTH))                 # distance to the next waypoint
WP_SPEED  = speed_profile(_ks, _ds)
WAYPOINTS = np.column_stack([CL.xs, CL.ys, CL.yaws, WP_SPEED]).astype(float)   # [x, y, yaw, speed]
np.savetxt("waypoints.csv", WAYPOINTS, delimiter=",")       # same format as Lab 8's waypoints.csv

fig, ax = plt.subplots(figsize=(10, 6))
track_ax(ax); lc = speed_line(ax, np.column_stack([CL_XY, WP_SPEED]), lw=3)
for k, (cx, cy) in enumerate(CORNER_XY):
    ax.annotate(f"C{k+1}", (cx, cy), color="red", fontsize=13, weight="bold")
fig.colorbar(lc, ax=ax, fraction=0.03, label="waypoint speed (m/s)")
ax.set_title("MPC waypoints: centerline with a safe speed profile")
plt.show()
print(f"Waypoints: {len(WAYPOINTS)} | speed {WP_SPEED.min():.1f} to {WP_SPEED.max():.1f} m/s | saved waypoints.csv")

---

# **Section 4 — The MPC (From Lab 8)** 🎛️

This is the Lab 8 kinematic MPC, with the ROS parts (subscribers, publishers, RViz markers)
removed so it runs directly on the gym. `control(x, y, yaw, v)` replaces `odom_callback` and
returns the same `[steer, speed]` the node published on `/drive`.

We made three small changes so it stays robust over thousands of solves with a residual on top. Each one is marked `[CHANGE n]` in the code:

| Change | Why |
|---|---|
| The sparse matrix pattern is fixed once, at a generic operating point | OSQP reuses its matrix between solves; a pattern that changes (an entry that is exactly 0 when $v = 0$) makes it fail |
| The linearization speed is kept at 0.1 m/s or more | The same reason: at a standstill some model entries are exactly 0 |
| The speed the QP starts from is clamped to `MAX_SPEED` | The residual can make the car faster than 6 m/s, which would make the QP's first state break its own speed limit |

`nearest_point` and `calc_interpolated_ref_trajectory` are the two helpers Lab 8 imported
from `utils.py`.

## 4.1 Configuration and Helpers

In [ ]:
# ── MPC configuration (Lab 8) ─────────────────────────────────────────────────
@dataclass
class mpc_config:
    NXK: int = 4  # length of kinematic state vector: z = [x, y, v, yaw]
    NU: int = 2  # length of input vector: u = = [steering speed, acceleration]
    TK: int = 8  # finite time horizon length kinematic

    # ---------------------------------------------------
    Rk: list = field(
        default_factory=lambda: np.diag([0.01, 100.0])
    )  # input cost matrix, penalty for inputs - [accel, steering_speed]
    Rdk: list = field(
        default_factory=lambda: np.diag([0.01, 100.0])
    )  # input difference cost matrix, penalty for change of inputs - [accel, steering_speed]
    Qk: list = field(
        default_factory=lambda: np.diag([13.5, 13.5, 5.5, 13.0])
    )  # state error cost matrix, for the the next (T) prediction time steps [x, y, v, yaw]
    Qfk: list = field(
        default_factory=lambda: np.diag([13.5, 13.5, 5.5, 13.0])
    )  # final state error matrix, penalty  for the final state constraints: [x, y, v, yaw]

    # ---------------------------------------------------

    N_IND_SEARCH: int = 20  # Search index number
    DTK: float = 0.1  # time step [s] kinematic
    dlk: float = 0.03  # dist step [m] kinematic
    LENGTH: float = 0.58  # Length of the vehicle [m]
    WIDTH: float = 0.31  # Width of the vehicle [m]
    WB: float = 0.33  # Wheelbase [m]
    MIN_STEER: float = -0.4189  # maximum steering angle [rad]
    MAX_STEER: float = 0.4189  # maximum steering angle [rad]
    MAX_DSTEER: float = np.deg2rad(180.0)  # maximum steering speed [rad/s]
    MAX_SPEED: float = 6.0  # maximum speed [m/s]
    MIN_SPEED: float = 0.0  # minimum backward speed [m/s]
    MAX_ACCEL: float = 3.0  # maximum acceleration [m/ss]


@dataclass
class State:
    x: float = 0.0
    y: float = 0.0
    delta: float = 0.0
    v: float = 0.0
    yaw: float = 0.0
    yawrate: float = 0.0
    beta: float = 0.0


# ── utils.py ──────────────────────────────────────────────────────────────────
def nearest_point(point, trajectory):
    """Closest point to `point` on the polyline `trajectory` (N, 2).
    Returns (projection, distance, t along the segment, segment index)."""
    diffs = trajectory[1:, :] - trajectory[:-1, :]
    l2s   = diffs[:, 0] ** 2 + diffs[:, 1] ** 2
    dots  = np.sum((point - trajectory[:-1, :]) * diffs, axis=1)
    t     = np.clip(dots / np.maximum(l2s, 1e-12), 0.0, 1.0)
    projections = trajectory[:-1, :] + (t * diffs.T).T
    dists = np.linalg.norm(point - projections, axis=1)
    i = int(np.argmin(dists))
    return projections[i], dists[i], t[i], i


def calc_interpolated_ref_trajectory(x, y, cx, cy, sp, cyaw, dt, T, vehicle_yaw=None):
    """Reference [x, y, v, yaw] for T+1 steps: start at the car's projection on the
    (closed) path and move along it by the reference speed times dt each step."""
    xs, ys = np.append(cx, cx[0]), np.append(cy, cy[0])
    vs     = np.append(sp, sp[0])
    yws    = np.unwrap(np.append(cyaw, cyaw[0]))
    seg = np.hypot(np.diff(xs), np.diff(ys))
    s   = np.concatenate([[0.0], np.cumsum(seg)])
    L   = s[-1]
    _, _, t, i = nearest_point(np.array([x, y]), np.column_stack([xs, ys]))
    s_t = s[i] + t * seg[i]
    ref = np.zeros((4, T + 1))
    for k in range(T + 1):
        ss = s_t % L
        ref[0, k] = np.interp(ss, s, xs)
        ref[1, k] = np.interp(ss, s, ys)
        ref[2, k] = np.interp(ss, s, vs)
        ref[3, k] = np.interp(ss, s, yws)
        s_t += max(ref[2, k], 0.5) * dt
    return ref


print("MPC config and utils defined.")

## 4.2 The MPC Controller

In [ ]:
# ── Kinematic MPC (Lab 8, ROS removed) ────────────────────────────────────────
class MPC:
    """
    Implement Kinematic MPC on the car.
    control(x, y, yaw, v) replaces odom_callback and returns (steer, speed, ref_path, prediction).
    """
    def __init__(self, waypoints, config=None):
        # waypoints: columns [x, y, yaw, speed] (Lab 8: loaded from waypoints.csv)
        self.waypoints = waypoints
        self.config = config or mpc_config()
        self.odelta_v = None
        self.odelta = None
        self.oa = None
        self.init_flag = 0

        # initialize MPC problem
        self.mpc_prob_init()

    def reset(self):
        """Forget the previous solution (start of a new episode)."""
        self.oa, self.odelta_v = None, None

    @staticmethod
    def _align_reference_yaw(ref_path, current_yaw):
        """
        Remove yaw wrapping discontinuity across +/-pi so MPC sees smooth heading errors.
        """
        ref_path_aligned = np.array(ref_path, copy=True)

        def wrap_to_pi(a):
            return math.atan2(math.sin(a), math.cos(a))

        ref_yaws = ref_path_aligned[3, :].astype(float)
        if ref_yaws.size == 0:
            return ref_path_aligned

        # First point: force shortest-angle difference to current yaw
        ref_yaws[0] = float(current_yaw) + wrap_to_pi(float(ref_yaws[0]) - float(current_yaw))

        # Remaining points: unwrap starting from that first aligned yaw
        if ref_yaws.size > 1:
            ref_yaws[1:] = np.unwrap(ref_yaws[1:], discont=np.pi, period=2 * np.pi)
            yaws_full = np.concatenate(([ref_yaws[0]], ref_yaws[1:]))
            ref_yaws = np.unwrap(yaws_full)[0:]

        ref_path_aligned[3, :] = ref_yaws
        return ref_path_aligned

    def control(self, x, y, yaw, v):
        """Same as odom_callback, minus ROS. Returns None if the QP fails."""

        # [CHANGE 3] the residual can push the car past MAX_SPEED, and a start state
        # above the speed limit makes the QP infeasible, so clamp it here
        v = min(max(v, self.config.MIN_SPEED), self.config.MAX_SPEED)
        vehicle_state = State(x=x, y=y, yaw=yaw, v=v)

        # Extract waypoint columns: [x, y, yaw, speed]
        ref_x   = self.waypoints[:, 0]
        ref_y   = self.waypoints[:, 1]
        ref_yaw = self.waypoints[:, 2]
        ref_v   = self.waypoints[:, 3]

        # Calculate the next reference trajectory for the next T steps
        ref_path = self.calc_ref_trajectory(vehicle_state, ref_x, ref_y, ref_yaw, ref_v)
        ref_path = self._align_reference_yaw(ref_path, vehicle_state.yaw)
        x0 = [vehicle_state.x, vehicle_state.y, vehicle_state.v, vehicle_state.yaw]

        # Solve the MPC control problem
        (
            self.oa,
            self.odelta_v,
            ox,
            oy,
            oyaw,
            ov,
            state_predict,
        ) = self.linear_mpc_control(ref_path, x0, self.oa, self.odelta_v)

        if self.oa is None or self.odelta_v is None:
            return None
        steer_output = float(np.clip(self.odelta_v[0], self.config.MIN_STEER, self.config.MAX_STEER))
        speed_output = vehicle_state.v + self.oa[0] * self.config.DTK
        return steer_output, float(speed_output), ref_path, (ox, oy)

    def mpc_prob_init(self):
        """
        Create MPC quadratic optimization problem using cvxpy, solver: OSQP
        Will be solved every iteration for control.
        """

        # Vehicle State Vector
        self.xk = cvxpy.Variable((self.config.NXK, self.config.TK + 1))

        # Control Input vector
        self.uk = cvxpy.Variable((self.config.NU, self.config.TK))
        objective = 0.0  # Objective value of the optimization problem
        constraints = []  # Create constraints array

        # Initialize reference vectors
        self.x0k = cvxpy.Parameter((self.config.NXK,))
        self.x0k.value = np.zeros((self.config.NXK,))

        # Initialize reference trajectory parameter
        self.ref_traj_k = cvxpy.Parameter((self.config.NXK, self.config.TK + 1))
        self.ref_traj_k.value = np.zeros((self.config.NXK, self.config.TK + 1))

        # Initializes block diagonal form of R = [R, R, ..., R] (NU*T, NU*T)
        R_block = block_diag(tuple([self.config.Rk] * self.config.TK))

        # Initializes block diagonal form of Rd = [Rd, ..., Rd] (NU*(T-1), NU*(T-1))
        Rd_block = block_diag(tuple([self.config.Rdk] * (self.config.TK - 1)))

        # Initializes block diagonal form of Q = [Q, Q, ..., Qf] (NX*T, NX*T)
        Q_block = [self.config.Qk] * (self.config.TK)
        Q_block.append(self.config.Qfk)
        Q_block = block_diag(tuple(Q_block))

        # Objective part 1: Penalize control inputs weighted by R
        objective += cvxpy.quad_form(cvxpy.vec(self.uk, order='F'), csc_matrix(R_block))

        # Objective part 2: Penalize state deviation from reference trajectory weighted by Q and Qf
        error = cvxpy.vec(self.xk - self.ref_traj_k, order='F')
        objective += cvxpy.quad_form(error, csc_matrix(Q_block))

        # Objective part 3: Penalize change in consecutive control inputs weighted by Rd
        delta_u = cvxpy.vec(self.uk[:, 1:] - self.uk[:, :-1], order='F')
        objective += cvxpy.quad_form(delta_u, csc_matrix(Rd_block))

        # Constraints 1: vehicle dynamics for the next T timesteps
        A_block = []
        B_block = []
        C_block = []

        # [CHANGE 1] build the sparsity pattern at a generic point (v, yaw != 0) so no
        # entry is dropped; every later solve reuses this same pattern
        path_predict = np.zeros((self.config.NXK, self.config.TK + 1))
        path_predict[2, :], path_predict[3, :] = 1.0, 0.5
        for t in range(self.config.TK):
            A, B, C = self.get_model_matrix(path_predict[2, t], path_predict[3, t], 0.0)
            A_block.append(A)
            B_block.append(B)
            C_block.extend(C)

        A_block = block_diag(tuple(A_block)).tocoo()
        B_block = block_diag(tuple(B_block)).tocoo()
        C_block = np.array(C_block)
        self._A_rc = (A_block.row, A_block.col)
        self._B_rc = (B_block.row, B_block.col)

        # [AA] Sparse matrix to CVX parameter for proper stuffing
        # Reference: https://github.com/cvxpy/cvxpy/issues/1159#issuecomment-718925710
        m, n = A_block.shape
        self.Annz_k = cvxpy.Parameter(A_block.nnz)
        data = np.ones(self.Annz_k.size)
        rows = A_block.row * n + A_block.col
        cols = np.arange(self.Annz_k.size)
        Indexer = csc_matrix((data, (rows, cols)), shape=(m * n, self.Annz_k.size))
        self.Annz_k.value = A_block.data
        self.Ak_ = cvxpy.reshape(Indexer @ self.Annz_k, (m, n), order="C")

        # Same as A
        m, n = B_block.shape
        self.Bnnz_k = cvxpy.Parameter(B_block.nnz)
        data = np.ones(self.Bnnz_k.size)
        rows = B_block.row * n + B_block.col
        cols = np.arange(self.Bnnz_k.size)
        Indexer = csc_matrix((data, (rows, cols)), shape=(m * n, self.Bnnz_k.size))
        self.Bk_ = cvxpy.reshape(Indexer @ self.Bnnz_k, (m, n), order="C")
        self.Bnnz_k.value = B_block.data

        # No need for sparse matrices for C as most values are parameters
        self.Ck_ = cvxpy.Parameter(C_block.shape)
        self.Ck_.value = C_block

        # Constraint part 1: Dynamics z_{t+1} = A*z_t + B*u_t + C
        constraints += [
            cvxpy.vec(self.xk[:, 1:], order='F') ==
            self.Ak_ @ cvxpy.vec(self.xk[:, :-1], order='F') +
            self.Bk_ @ cvxpy.vec(self.uk, order='F') +
            self.Ck_
        ]

        # Constraint part 2: Steering rate limit
        constraints += [
            cvxpy.abs(self.uk[1, 1:] - self.uk[1, :-1]) <=
            self.config.MAX_DSTEER * self.config.DTK
        ]

        # Constraint part 3: State/input bounds and initial state
        constraints += [
            self.xk[:, 0] == self.x0k,                      # initial state
            self.xk[2, :] <= self.config.MAX_SPEED,          # speed upper bound
            self.xk[2, :] >= self.config.MIN_SPEED,          # speed lower bound
            self.uk[0, :] <= self.config.MAX_ACCEL,          # accel upper bound
            self.uk[0, :] >= -self.config.MAX_ACCEL,         # accel lower bound
            self.uk[1, :] <= self.config.MAX_STEER,          # steer upper bound
            self.uk[1, :] >= self.config.MIN_STEER,          # steer lower bound
        ]

        # Optimization goal: minimize the objective function
        self.MPC_prob = cvxpy.Problem(cvxpy.Minimize(objective), constraints)

    def calc_ref_trajectory(self, state, cx, cy, cyaw, sp):
        """Interpolated reference trajectory for T steps: [x, y, v, yaw]."""
        return calc_interpolated_ref_trajectory(
            state.x, state.y,
            cx, cy, sp, cyaw,
            self.config.DTK,
            self.config.TK,
            vehicle_yaw=state.yaw,
        )

    def predict_motion(self, x0, oa, od, xref):
        path_predict = xref * 0.0
        for i, _ in enumerate(x0):
            path_predict[i, 0] = x0[i]

        state = State(x=x0[0], y=x0[1], yaw=x0[3], v=x0[2])
        for (ai, di, i) in zip(oa, od, range(1, self.config.TK + 1)):
            state = self.update_state(state, ai, di)
            path_predict[0, i] = state.x
            path_predict[1, i] = state.y
            path_predict[2, i] = state.v
            path_predict[3, i] = state.yaw

        return path_predict

    def update_state(self, state, a, delta):
        # input check
        if delta >= self.config.MAX_STEER:
            delta = self.config.MAX_STEER
        elif delta <= -self.config.MAX_STEER:
            delta = -self.config.MAX_STEER

        state.x = state.x + state.v * math.cos(state.yaw) * self.config.DTK
        state.y = state.y + state.v * math.sin(state.yaw) * self.config.DTK
        state.yaw = (
            state.yaw + (state.v / self.config.WB) * math.tan(delta) * self.config.DTK
        )
        state.v = state.v + a * self.config.DTK

        if state.v > self.config.MAX_SPEED:
            state.v = self.config.MAX_SPEED
        elif state.v < self.config.MIN_SPEED:
            state.v = self.config.MIN_SPEED

        return state

    def get_model_matrix(self, v, phi, delta):
        """
        Linear, discrete-time kinematic model: z_{t+1} = A z_t + B u_t + C, z = [x, y, v, yaw]
        """
        v = max(v, 0.1)   # [CHANGE 2] keep the model entries non-zero so the solver sparsity stays fixed

        # State (or system) matrix A, 4x4
        A = np.zeros((self.config.NXK, self.config.NXK))
        A[0, 0] = 1.0
        A[1, 1] = 1.0
        A[2, 2] = 1.0
        A[3, 3] = 1.0
        A[0, 2] = self.config.DTK * math.cos(phi)
        A[0, 3] = -self.config.DTK * v * math.sin(phi)
        A[1, 2] = self.config.DTK * math.sin(phi)
        A[1, 3] = self.config.DTK * v * math.cos(phi)
        A[3, 2] = self.config.DTK * math.tan(delta) / self.config.WB

        # Input Matrix B; 4x2
        B = np.zeros((self.config.NXK, self.config.NU))
        B[2, 0] = self.config.DTK
        B[3, 1] = self.config.DTK * v / (self.config.WB * math.cos(delta) ** 2)

        C = np.zeros(self.config.NXK)
        C[0] = self.config.DTK * v * math.sin(phi) * phi
        C[1] = -self.config.DTK * v * math.cos(phi) * phi
        C[3] = -self.config.DTK * v * delta / (self.config.WB * math.cos(delta) ** 2)

        return A, B, C

    def mpc_prob_solve(self, ref_traj, path_predict, x0):
        self.x0k.value = x0

        A_block = []
        B_block = []
        C_block = []
        for t in range(self.config.TK):
            A, B, C = self.get_model_matrix(path_predict[2, t], path_predict[3, t], 0.0)
            A_block.append(A)
            B_block.append(B)
            C_block.extend(C)

        # [CHANGE 1] read the values at the fixed sparsity pattern
        A_full = block_diag(tuple(A_block)).toarray()
        B_full = block_diag(tuple(B_block)).toarray()
        self.Annz_k.value = A_full[self._A_rc]
        self.Bnnz_k.value = B_full[self._B_rc]
        self.Ck_.value = np.array(C_block)

        self.ref_traj_k.value = ref_traj

        # Solve the optimization problem in CVXPY
        self.MPC_prob.solve(solver=cvxpy.OSQP, verbose=False, warm_start=True)

        if (
            self.MPC_prob.status == cvxpy.OPTIMAL
            or self.MPC_prob.status == cvxpy.OPTIMAL_INACCURATE
        ):
            ox = np.array(self.xk.value[0, :]).flatten()
            oy = np.array(self.xk.value[1, :]).flatten()
            ov = np.array(self.xk.value[2, :]).flatten()
            oyaw = np.array(self.xk.value[3, :]).flatten()
            oa = np.array(self.uk.value[0, :]).flatten()
            odelta = np.array(self.uk.value[1, :]).flatten()
        else:
            oa, odelta, ox, oy, oyaw, ov = None, None, None, None, None, None

        return oa, odelta, ox, oy, oyaw, ov

    def linear_mpc_control(self, ref_path, x0, oa, od):
        """
        MPC control with updating operational point iteratively
        """
        if oa is None or od is None:
            oa = [0.0] * self.config.TK
            od = [0.0] * self.config.TK

        # Predict the vehicle motion for T steps
        path_predict = self.predict_motion(x0, oa, od, ref_path)

        # Create and solve the optimization problem
        mpc_a, mpc_delta, mpc_x, mpc_y, mpc_yaw, mpc_v = self.mpc_prob_solve(
            ref_path, path_predict, x0
        )

        return mpc_a, mpc_delta, mpc_x, mpc_y, mpc_yaw, mpc_v, path_predict


_test = MPC(WAYPOINTS)
_out  = _test.control(float(CL.xs[0]), float(CL.ys[0]), float(CL.yaws[0]), 0.0)
print(f"MPC built and solved once: steer = {_out[0]:+.3f} rad, speed = {_out[1]:.2f} m/s")

---

# **Section 5 — The Residual Environment** 🧩

## 5.1 Observation, Action and Reward

**Observation** (114 values, all scaled to between 0 and 1). The first 110 values are
**exactly the observation from Learn to Drive and Learn to Race**; the last 4 are new:

| Index | What it is | How it is scaled |
|---|---|---|
| 0 to 107 | Lidar scan (108 beams) | Divided by 10 m |
| 108 | Speed | Divided by the top speed (8 m/s) |
| 109 | Steering angle | Full left = 0, straight = 0.5, full right = 1 |
| 110 | **MPC steering command** | Same scaling as the steering angle |
| 111 | **MPC speed command** | Divided by the top speed |
| 112 | **Sideways tracking error** to the waypoints | −1.1 to +1.1 m mapped to 0 to 1 |
| 113 | **Heading error** to the waypoints | −π/4 to +π/4 mapped to 0 to 1 |

The policy needs to know *what the MPC is about to do* and *how well it's tracking* to decide
whether to correct it.

**Action:** the same two numbers in $[-1, 1]$ as before, now a **correction** added to the MPC
command: up to ±0.1 rad of steering and ±2 m/s of speed. The sum is clipped to the car's
limits (±0.4189 rad, 0 to 8 m/s). With the residual at zero, the car drives exactly the MPC.

**Reward:**

| Term | Value | Why |
|---|---|---|
| **Progress** | +1.0 per metre gained along the centerline | Fixed-length episodes, so more metres means a faster lap |
| **Wall penalty** | −0.5 per metre that the car is inside 0.4 m of a wall | Same term as Learn to Race |
| **Residual penalty** (new) | −0.05 × (a₀² + a₁²) | Only override the MPC where it pays off |
| **Crash penalty** | −10, and the episode ends | Hitting a wall, or a spin-out (sideways acceleration past 11.3 m/s²) |

Training episodes start at a **random point on the track** and last 30 s (300 decisions);
evaluation laps always start from the start/finish line.

## 5.2 Implementation

In [ ]:
# ── Residual configuration ────────────────────────────────────────────────────
RES_STEER_MAX  = 0.10    # rad, largest steering correction
RES_SPEED_MAX  = 2.0     # m/s, largest speed correction
V_MAX          = 8.0     # m/s, top speed of MPC + residual together
SIM_STEPS      = 10      # sim steps per decision (0.1 s, same as DTK)
EPISODE_STEPS  = 300     # decisions per training episode (30 s)

# Tyre grip (same rule as Learn the Racing Line)
MU_G           = 1.0489 * 9.81   # grip limit mu*g, about 10.3 m/s^2
SPIN_OUT_LIMIT = 1.1 * MU_G      # past this we call it a spin-out
A_LAT_FILTER   = 0.05            # low-pass filter on lateral accel (~0.2 s)

# Reward coefficients (the sweep in 7.3 tunes CRASH_PENALTY and RESIDUAL_COEF)
PROGRESS_COEF       = 1.0    # progress reward, per metre gained along the centerline
WALL_PROX_COEF      = 0.5    # wall penalty, per metre inside the threshold
WALL_PROX_THRESHOLD = 0.4    # metres
RESIDUAL_COEF       = 0.05   # residual penalty on the squared correction
CRASH_PENALTY       = 10.0   # crash penalty (wall hit or spin-out)


class ResidualMPCWrapper(gym.Env):
    """
    Observation (114-dim, all in [0,1]): lidar(108), speed, steer, MPC steer, MPC speed,
                                          sideways tracking error, heading error
    Action (2-dim, in [-1,1]):  [0] steering correction, [1] speed correction
    Each step() = one MPC decision: u = u_MPC + u_RL held for SIM_STEPS sim steps.
    """

    def __init__(self, map_name=MAP_NAME, num_beams=108, render_mode=None):
        super().__init__()
        from f1tenth_gym.envs.env_config import (
            EnvConfig, SimulationConfig, ObservationConfig, ResetConfig,
        )
        from f1tenth_gym.envs.lidar.config import LiDARConfig
        from f1tenth_gym.envs.observation import ObservationType
        from f1tenth_gym.envs.reset import ResetStrategy
        from f1tenth_gym.envs.integrators import IntegratorType
        from f1tenth_gym.envs.dynamic_models import DynamicModel

        env_config = EnvConfig(
            map_name=map_name,
            num_agents=1,
            simulation_config=SimulationConfig(
                timestep=0.01,
                integrator=IntegratorType.RK4,
                dynamics_model=DynamicModel.ST,
                max_laps=100,  # we end episodes ourselves
            ),
            observation_config=ObservationConfig(type=ObservationType.DIRECT),
            lidar_config=LiDARConfig(num_beams=num_beams, range_max=10.0, noise_std=0.0),
            reset_config=ResetConfig(strategy=ResetStrategy.RL_RANDOM_STATIC),
            render_enabled=(render_mode is not None),
        )
        self.env = gym.make("f1tenth_gym:f1tenth-v0", config=env_config, render_mode=render_mode)

        self.track      = self.env.unwrapped.track
        self.track_len  = float(self.track.centerline.length)
        self.num_beams  = num_beams
        self.lidar_clip = 10.0
        self.max_steer  = float(self.env.action_space.high[0][0])
        self.max_steps  = EPISODE_STEPS
        self.dt         = 0.01
        self.mpc        = MPC(WAYPOINTS)   # the Lab 8 controller
        self._wp_xy     = np.vstack([WAYPOINTS[:, :2], WAYPOINTS[:1, :2]])

        self.observation_space = gym.spaces.Box(low=0.0, high=1.0, shape=(num_beams + 6,), dtype=np.float32)
        self.action_space      = gym.spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)

    # ----------------------------------------------------------

    def _solve_mpc(self):
        """MPC command [steer, speed] for the current state (gentle fallback if the QP fails)."""
        st = self.raw_obs["agent_0"]["std_state"]
        out = self.mpc.control(float(st[0]), float(st[1]), float(st[4]), float(st[3]))
        if out is None:
            self.mpc_failures += 1
            self.mpc.reset()
            return 0.0, max(float(st[3]) - 0.3, 1.0)
        return out[0], out[1]

    def _tracking_errors(self, state):
        """Signed sideways distance (+ = left) and heading error to the waypoint path."""
        x, y, yaw = float(state[0]), float(state[1]), float(state[4])
        proj, dist, _, i = nearest_point(np.array([x, y]), self._wp_xy)
        ref_yaw = WAYPOINTS[i % len(WAYPOINTS), 2]
        side = np.cos(ref_yaw) * (y - proj[1]) - np.sin(ref_yaw) * (x - proj[0])
        ey   = float(np.sign(side) * dist)
        epsi = float(math.atan2(math.sin(yaw - ref_yaw), math.cos(yaw - ref_yaw)))
        return ey, epsi

    def _process_obs(self, raw_obs):
        agent = raw_obs["agent_0"]
        scan  = np.array(agent["scan"], dtype=np.float32) / self.lidar_clip
        state = agent["std_state"]
        speed_n = np.clip(float(state[3]) / V_MAX, 0.0, 1.0)
        steer_n = np.clip((float(state[2]) + self.max_steer) / (2 * self.max_steer), 0.0, 1.0)

        # ═════════════════════════════════════════════════════════════
        # ✏️  TODO 2 — Residual Observation: MPC Command and Tracking Errors
        # ═════════════════════════════════════════════════════════════

        # Scale the MPC's current command (self.mpc_cmd = steer, speed) the same
        # way as the car's own steering and speed -> mpc_steer_n, mpc_speed_n.
        # Get the tracking errors with self._tracking_errors(state) and map them
        # to [0, 1]: ey over [-HALF_WIDTH, HALF_WIDTH] -> ey_n, epsi over
        # [-pi/4, pi/4] -> epsi_n (clip both).

        # ─────────────────────────────────────────────────────────────





        # ═════════════════════════════════════════════════════════════
        return np.concatenate([
            scan, np.array([speed_n, steer_n, mpc_steer_n, mpc_speed_n, ey_n, epsi_n], dtype=np.float32)
        ]).astype(np.float32)

    def _apply_residual(self, action):

        # ═════════════════════════════════════════════════════════════
        # ✏️  TODO 3 — Add the Residual to the MPC Command
        # ═════════════════════════════════════════════════════════════

        # action[0], action[1] are in [-1, 1]. Scale them to a steering
        # correction (RES_STEER_MAX rad) and a speed correction (RES_SPEED_MAX
        # m/s), add them to the MPC command self.mpc_cmd, and clip the result to
        # the car's limits: steering to +-self.max_steer, speed to [0, V_MAX].
        # Return: steer, speed

        # ─────────────────────────────────────────────────────────────





        # ═════════════════════════════════════════════════════════════
        pass  # remove once you've implemented the steps above

    # ----------------------------------------------------------

    def reset(self, seed=None, options=None):
        raw_obs, info     = self.env.reset(seed=seed, options=options)
        self.raw_obs      = raw_obs
        self.current_step = 0
        self.prev_s       = float(raw_obs["agent_0"]["frenet_pose"][0])
        self.a_lat        = 0.0         # filtered sideways acceleration (m/s^2)
        self.mpc_failures = 0
        self.path         = []          # (x, y, speed, lateral accel) every sim step, for the plots
        self.residuals    = []          # (steer correction, speed correction) every decision
        self.mpc.reset()
        self.mpc_cmd      = self._solve_mpc()
        return self._process_obs(raw_obs), info

    def step(self, action):
        action = np.asarray(action, dtype=np.float32)
        steer, speed = self._apply_residual(action)
        self.residuals.append((steer - self.mpc_cmd[0], speed - self.mpc_cmd[1]))
        delta_progress, collision, spun_out, done, truncated = 0.0, False, False, False, False

        # hold the command for one MPC period (SIM_STEPS sim steps)
        for _ in range(SIM_STEPS):
            self.raw_obs, _, done, truncated, info = self.env.step(
                np.array([[steer, speed]], dtype=np.float32)
            )
            agent = self.raw_obs["agent_0"]
            s_new = float(agent["frenet_pose"][0])
            ds = s_new - self.prev_s
            if ds < -self.track_len / 2: ds += self.track_len     # lap wrap-around
            if ds >  self.track_len / 2: ds -= self.track_len
            delta_progress += ds
            self.prev_s = s_new
            st = agent["std_state"]
            a_lat_now = abs(float(st[3]) * float(st[5]))          # lateral accel = speed x yaw rate
            self.a_lat += A_LAT_FILTER * (a_lat_now - self.a_lat)
            self.path.append((float(st[0]), float(st[1]), float(st[3]), self.a_lat))
            collision = bool(agent.get("collision", False))
            spun_out  = self.a_lat > SPIN_OUT_LIMIT               # past the grip limit
            if collision or spun_out or done:
                break
        self.current_step += 1
        min_scan = float(np.min(self.raw_obs["agent_0"]["scan"]))

        # ═════════════════════════════════════════════════════════════
        # ✏️  TODO 4 — Residual Reward
        # ═════════════════════════════════════════════════════════════

        # Build the reward from the four terms in Section 5.1:
        #   progress reward  : delta_progress scaled by PROGRESS_COEF
        #   wall penalty     : grows as min_scan drops below WALL_PROX_THRESHOLD
        #   residual penalty : RESIDUAL_COEF times the squared size of the action
        #   crash penalty    : on a collision or a spin-out, the reward becomes
        #                      -CRASH_PENALTY and the episode ends (done = True)

        # ─────────────────────────────────────────────────────────────
        reward = 0.0





        # ═════════════════════════════════════════════════════════════
        if self.current_step >= self.max_steps:
            truncated = True

        if not (collision or spun_out):
            self.mpc_cmd = self._solve_mpc()   # MPC command for the next decision

        info = {
            "speed":     float(self.raw_obs["agent_0"]["std_state"][3]),
            "collision": collision,
            "spun_out":  spun_out,
            "ds":        delta_progress,
            "min_scan":  min_scan,
            "res_steer": self.residuals[-1][0],
            "res_speed": self.residuals[-1][1],
        }
        return self._process_obs(self.raw_obs), float(reward), bool(done), bool(truncated), info

    def render(self):
        return self.env.render()

    def close(self):
        self.env.close()


print("ResidualMPCWrapper defined!")

## 5.3 Baseline: the MPC Alone

With the residual at zero, the wrapper drives **exactly the MPC**. That is our baseline, and
it is also what the untrained policy does on average (its mean action is zero).

We use `run_lap` for all evaluation from here on. It drives one lap from the start line with
any policy and returns the lap time, the `(x, y, speed, lateral accel)` trace, and the
residual applied at every decision. A lap that crashes is marked **DNF** (did
not finish).

In [ ]:
# ── Lap timing + the MPC-alone baseline ───────────────────────────────────────
START_POSE = np.array([[CL.xs[0], CL.ys[0], CL.yaws[0]]], dtype=np.float64)


def run_lap(env, policy_fn, max_decisions=2000):
    """Drive ONE lap from the start line. policy_fn(obs) -> action in [-1, 1]^2.
    Returns dict(lap_time [s, or None for a DNF], path (N,4), residuals (M,2), crashed, dist [m])."""
    obs, _ = env.reset(options={"poses": START_POSE})
    env.max_steps = 10**9   # no time limit while timing a lap
    dist, n, crashed = 0.0, 0, False
    while dist < env.track_len and n < max_decisions:
        obs, _, _, _, info = env.step(policy_fn(obs))
        dist += info["ds"]; n += 1
        if info["collision"] or info["spun_out"]:
            crashed = True
            break
    env.max_steps = EPISODE_STEPS
    done_lap = (not crashed) and dist >= env.track_len
    return {"lap_time": len(env.path) * env.dt if done_lap else None,
            "path": np.array(env.path), "residuals": np.array(env.residuals),
            "crashed": crashed, "dist": dist}


def lap_str(r):
    return f"{r['lap_time']:.2f} s" if r["lap_time"] else f"DNF after {r['dist']:.0f} m"


def zero_residual(obs):
    return np.zeros(2, dtype=np.float32)


base_env = ResidualMPCWrapper()
t0 = time.time()
BASELINE = run_lap(base_env, zero_residual)
print(f"MPC alone: {lap_str(BASELINE)}  (average speed {np.mean(BASELINE['path'][:, 2]):.2f} m/s, "
      f"{time.time() - t0:.0f} s of compute)")

fig, ax = plt.subplots(figsize=(10, 6))
track_ax(ax); lc = speed_line(ax, BASELINE["path"], lw=2.2)
fig.colorbar(lc, ax=ax, fraction=0.03, label="speed (m/s)")
ax.set_title(f"MPC alone: {lap_str(BASELINE)}")
plt.show()

---

# **Section 6 — PPO Algorithm (Unchanged)** 🧠

The network, GAE, and clipped loss are identical to our from-scratch PPO in Learn to
Drive; only the environment changed.

> **📌 Note:** Since this is unchanged from your earlier notebooks, feel free to paste in your own working PPO implementation (network, GAE, clipped loss) from Learn to Drive here instead of rewriting it, if you already have one you trust.

In [ ]:
# ── Actor-Critic Network ──────────────────────────────────────────────────────
# Separate actor and critic MLPs (64x64 each, nothing shared).
# The small init gain on the action head keeps the first actions close to 0,
# so the untrained policy drives like the plain MPC.

def _orthogonal_init(layer, gain):
    nn.init.orthogonal_(layer.weight, gain=gain)
    nn.init.constant_(layer.bias, 0.0)

class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()

        # actor
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.actor_mean = nn.Linear(hidden, act_dim)

        # critic
        self.critic_body = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.critic = nn.Linear(hidden, 1)

        # learnable log std (same for every state)
        self.log_std = nn.Parameter(torch.zeros(act_dim))

        # orthogonal init: sqrt(2) for hidden layers, 0.01 for the action head, 1.0 for the value head
        for layer in self.actor:
            if isinstance(layer, nn.Linear): _orthogonal_init(layer, gain=2 ** 0.5)
        for layer in self.critic_body:
            if isinstance(layer, nn.Linear): _orthogonal_init(layer, gain=2 ** 0.5)
        _orthogonal_init(self.actor_mean, gain=0.01)
        _orthogonal_init(self.critic, gain=1.0)

    def forward(self, obs):
        # ═══════════════════════════════════════════════════════════════
        # ✏️  TODO 5 — Actor-Critic Forward Pass
        # ═══════════════════════════════════════════════════════════════

        # Pass obs through the actor to get the action mean and through the
        # critic to get the state value, then build a Normal distribution
        # from the mean and the learned log-std.
        # Return: dist, value

        # ───────────────────────────────────────────────────────────────





        # ═══════════════════════════════════════════════════════════════
        pass  # remove once you've implemented the steps above

    @torch.no_grad()
    def act(self, obs, deterministic=False):
        """Sample an action (or take the mean if deterministic)."""
        dist, value = self.forward(obs)
        action = dist.mean if deterministic else dist.sample()
        log_prob = dist.log_prob(action).sum(-1)
        return action, log_prob, value


def greedy_policy(net):
    """Deterministic policy function for run_lap()."""
    def _pi(obs):
        with torch.no_grad():
            a, _, _ = net.act(torch.tensor(obs, dtype=torch.float32, device=next(net.parameters()).device),
                              deterministic=True)
        return np.clip(a.cpu().numpy(), -1.0, 1.0)
    return _pi


print('ActorCritic defined.')

## 6.1 Environment Builders

In [ ]:
# ── Environment factory ───────────────────────────────────────────────────────

def make_env(rank, seed=0):
    """Returns a function that creates one ResidualMPCWrapper instance (with its own MPC)."""
    def _init():
        env = ResidualMPCWrapper()
        env.reset(seed=seed + rank)   # different random start per env
        return env
    return _init


def make_training_env(num_envs):
    fns = [make_env(i) for i in range(num_envs)]
    env = DummyVecEnv(fns) if num_envs == 1 else SubprocVecEnv(fns)
    env = VecMonitor(env)
    env = VecNormalize(env, norm_obs=False, norm_reward=True, clip_reward=10.0, gamma=0.99)
    return env


print("Env builders defined.")

## 6.2 GAE: Generalized Advantage Estimation

In [ ]:
# ── GAE — Generalized Advantage Estimation ────────────────────────────────────
def compute_gae(rewards, values, dones, last_value, gamma=0.99, lam=0.95):
    """
    rewards, values, dones : tensors of shape (T, N)  [T steps, N envs]
    last_value             : tensor (N,), V(s_T) for bootstrapping
    Returns advantages and returns, both (T, N).
    """
    T, N = rewards.shape
    advantages = torch.zeros_like(rewards)
    last_adv = torch.zeros(N, device=rewards.device)

    # ═══════════════════════════════════════════════════════════════
    # ✏️  TODO 6 — Compute GAE Advantages
    # ═══════════════════════════════════════════════════════════════

    # Loop backwards from t = T-1 to 0. At each step, take the next value
    # (last_value at the end), discount it only if the episode didn't end,
    # compute the TD residual, then add the decayed advantage from step t+1
    # and store the result in advantages[t].

    # ───────────────────────────────────────────────────────────────





    # ═══════════════════════════════════════════════════════════════

    returns = advantages + values
    return advantages, returns


print('compute_gae defined.')

---

# **Section 7 — Training** 🏋️

One "step" here is one **MPC decision** (0.1 s of driving). Each decision solves a small QP, so
steps are slower than in the earlier notebooks, but a residual policy needs far fewer of them.

| Addition | Why |
|---|---|
| **Small initial exploration** (policy std starts at 0.37 instead of 1.0) | Early corrections stay small, so the car drives like the MPC while it learns |
| **Learning-rate and entropy annealing** (both down to 0) | Explore early, then commit to one set of corrections |
| **Best-lap checkpoint** | The network with the fastest completed eval lap is saved and used for evaluation |

In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
NUM_ENVS    = 4             # parallel environments (each runs its own MPC)
TOTAL_STEPS = 200_000       # MPC decisions (about 5.5 hours of driving, roughly 30 to 40 min on Colab)
DEVICE      = "cuda"
EVAL_FREQ   = 10_000        # a timed eval lap every N decisions; these laps build the plots

print("Training config:")
print(f"  Map:    {MAP_NAME}")
print(f"  Envs:   {NUM_ENVS}")
print(f"  Steps:  {TOTAL_STEPS:,} decisions")
print(f"  Device: {DEVICE}")

## 7.2 WandB Login

In [ ]:
# ── WandB login ───────────────────────────────────────────────────────────────
# Choose option 1 or 2, then paste your WANDB_API key
# Get it from: wandb.ai/authorize
# WANDB_API_KEY = "your_key_here"
# wandb.login(key=WANDB_API_KEY)

wandb.login()

## 7.3 Reward Sweep *(Run Before Full Training - Optional)*

| Knob | Too low | Too high |
|---|---|---|
| `CRASH_PENALTY` | Reckless corrections, frequent crashes | Timid: the policy barely leaves the MPC |
| `RESIDUAL_COEF` | Large, noisy corrections everywhere | The policy is pinned to the MPC and cannot improve it |

The sweep trains a short policy for each combination and ranks them by **lap time**. We don't
rank by reward, because the swept coefficients are part of the reward itself. A DNF lap scores
`SWEEP_FAIL_TIME`.

**How to use:**
1. Complete **TODOs 1 to 6** first; the sweep uses your wrapper and your PPO pieces
2. Edit the `parameters` lists below to the values you want to compare
3. Run the cell; WandB launches one run per combination
4. Open the sweep link that prints; results update live
5. Put the best values in the configuration cell (Section 5.2), re-run it, then train in 7.5

In [ ]:
# ── WandB Reward Sweep ────────────────────────────────────────────────────────
sweep_config = {
    "method": "grid",
    "metric": {"name": "eval/lap_time", "goal": "minimize"},
    "parameters": {
        "crash_penalty": {"values": [5.0, 10.0]},
        "residual_coef": {"values": [0.02, 0.1]},
    }
}

SWEEP_STEPS     = 20_000
SWEEP_ENVS      = 2
SWEEP_N_STEPS   = 512       # rollout length per env per update
SWEEP_EPOCHS    = 10
SWEEP_MINIBATCH = 256
SWEEP_FAIL_TIME = 200.0     # score for a DNF lap


def sweep_train():
    global CRASH_PENALTY, RESIDUAL_COEF

    run = wandb.init(reinit=True)  # new run for each combination
    cfg = wandb.config
    print(f"  crash_penalty={cfg.crash_penalty} | residual_coef={cfg.residual_coef}")

    # step() reads these globals, so set them before building the envs
    _orig = (CRASH_PENALTY, RESIDUAL_COEF)
    CRASH_PENALTY = cfg.crash_penalty
    RESIDUAL_COEF = cfg.residual_coef

    s_train = s_eval = None
    try:
        torch.manual_seed(42); np.random.seed(42)
        device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")

        # DummyVecEnv keeps the envs in this process, so they see the globals set above
        s_train = DummyVecEnv([make_env(i) for i in range(SWEEP_ENVS)])
        s_train = VecMonitor(s_train)
        s_train = VecNormalize(s_train, norm_obs=False, norm_reward=True,
                               clip_reward=10.0, gamma=0.99)
        s_eval  = ResidualMPCWrapper()

        obs_dim = s_train.observation_space.shape[0]
        act_dim = s_train.action_space.shape[0]
        net = ActorCritic(obs_dim, act_dim).to(device)
        net.log_std.data.fill_(-1.0)                       # start with small corrections
        opt = optim.Adam(net.parameters(), lr=3e-4, eps=1e-5)

        obs = torch.tensor(s_train.reset(), dtype=torch.float32, device=device)
        for _update in range(SWEEP_STEPS // (SWEEP_N_STEPS * SWEEP_ENVS)):
            b_obs_l, b_act_l, b_logp_l, b_val_l, b_rew_l, b_done_l = [], [], [], [], [], []
            for _t in range(SWEEP_N_STEPS):
                action, logp, value = net.act(obs)
                next_obs, reward, dones, _ = s_train.step(np.clip(action.cpu().numpy(), -1.0, 1.0))
                b_obs_l.append(obs); b_act_l.append(action)
                b_logp_l.append(logp); b_val_l.append(value)
                b_rew_l.append(torch.tensor(reward, dtype=torch.float32, device=device))
                b_done_l.append(torch.tensor(dones, dtype=torch.float32, device=device))
                obs = torch.tensor(next_obs, dtype=torch.float32, device=device)

            with torch.no_grad():
                _, last_value = net(obs)
            values = torch.stack(b_val_l)
            advantages, returns = compute_gae(torch.stack(b_rew_l), values, torch.stack(b_done_l),
                                              last_value, gamma=0.99, lam=0.95)
            b_obs  = torch.stack(b_obs_l).reshape(-1, obs_dim)
            b_act  = torch.stack(b_act_l).reshape(-1, act_dim)
            b_logp = torch.stack(b_logp_l).reshape(-1)
            b_adv  = advantages.reshape(-1)
            b_ret  = returns.reshape(-1)

            idxs = np.arange(b_obs.shape[0])
            for _epoch in range(SWEEP_EPOCHS):
                np.random.shuffle(idxs)
                for start in range(0, len(idxs), SWEEP_MINIBATCH):
                    mb = idxs[start:start + SWEEP_MINIBATCH]
                    dist, value = net(b_obs[mb])
                    new_logp = dist.log_prob(b_act[mb]).sum(-1)
                    entropy  = dist.entropy().sum(-1).mean()
                    mb_adv   = b_adv[mb]
                    mb_adv   = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)
                    ratio    = torch.exp(new_logp - b_logp[mb])
                    policy_loss = -torch.min(ratio * mb_adv,
                                             torch.clamp(ratio, 0.8, 1.2) * mb_adv).mean()
                    value_loss  = ((value - b_ret[mb]) ** 2).mean()
                    loss = policy_loss + 0.5 * value_loss - 0.001 * entropy
                    opt.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(net.parameters(), 0.5)
                    opt.step()

        # one timed lap is the sweep metric
        lap = run_lap(s_eval, greedy_policy(net))
        lap_time = lap["lap_time"] if lap["lap_time"] is not None else SWEEP_FAIL_TIME
        wandb.log({
            "eval/lap_time":      lap_time,
            "eval/lap_completed": float(lap["lap_time"] is not None),
            "eval/lap_distance":  lap["dist"],
            "eval/avg_speed":     float(np.mean(lap["path"][:, 2])) if len(lap["path"]) else 0.0,
        })
        print(f"    lap: {lap_str(lap)}")
    finally:
        if s_train is not None: s_train.close()
        if s_eval  is not None: s_eval.close()
        CRASH_PENALTY, RESIDUAL_COEF = _orig   # restore your chosen values
        run.finish()

# ── Launch ───────────────────────────────────────────────────────────────────
n_combos = (len(sweep_config["parameters"]["crash_penalty"]["values"]) *
            len(sweep_config["parameters"]["residual_coef"]["values"]))
sweep_id = wandb.sweep(sweep_config, project="f1tenth-rl-residual-mpc-sweep")
print(f"Sweep: {n_combos} combinations of {SWEEP_STEPS:,} decisions each")
print(f"View at: https://wandb.ai/{wandb.Api().default_entity}/f1tenth-rl-residual-mpc-sweep/sweeps/{sweep_id}")
wandb.agent(sweep_id, function=sweep_train, count=None)
print()
print("✅ Sweep complete. Open the WandB link above to compare results.")
print("   Put the best values in the configuration cell (Section 5.2), re-run it, then train in 7.5.")

## 7.4 Training Dashboard

| Panel | What to look for |
|---|---|
| **ep_reward** | Rises as the car covers more ground per episode |
| **crash rate (last 100 episodes)** | Stays low: the MPC keeps the car on track while the policy explores |
| **eval lap time** | Falls below the dashed MPC-alone line; red crosses mark DNF eval laps |
| **avg speed** | Rises above what the MPC drives by itself |
| **mean residual size** | How hard the policy overrides the MPC; it should settle, not keep growing |
| **total loss** | Stays bounded; in RL the loss is not a progress score, judge by lap time |

In [ ]:
# ── Live Inline Training Plots ────────────────────────────────────────────────
_live = {"steps": [], "rew": [], "crash": [], "speed": [], "loss": [], "residual": []}
LAP_HISTORY = []    # one entry per eval lap: dict(step, lap_time, dist, path, residuals)


def draw_live_plots(total_steps):
    if len(_live["steps"]) < 2:
        return
    BG = "#0d1117"
    fig = plt.figure(figsize=(15, 7), facecolor=BG)
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.3)
    panels = [
        (gs[0, 0], _live["rew"],      "ep_reward ↑",                     "#00bcd4"),
        (gs[0, 1], _live["crash"],    "crash rate, last 100 episodes ↓", "#f44336"),
        (gs[1, 0], _live["speed"],    "avg_speed m/s ↑",                 "#69ff47"),
        (gs[1, 1], _live["residual"], "mean residual size |a|",          "#9c27b0"),
        (gs[1, 2], _live["loss"],     "total_loss",                      "#e91e63"),
    ]
    for spec, data, title, color in panels:
        ax = fig.add_subplot(spec); ax.set_facecolor("#1a1a2e")
        ax.plot(_live["steps"], data, color=color, lw=2, marker="o", markersize=3)
        ax.fill_between(_live["steps"], data, min(data), alpha=0.15, color=color)
        ax.set_title(title, color="white", fontsize=11, pad=4)
        ax.set_xlabel("Decisions", color="#aaa", fontsize=9)
        ax.tick_params(colors="#aaa", labelsize=9)
        for spine in ax.spines.values(): spine.set_edgecolor("#333")

    ax = fig.add_subplot(gs[0, 2]); ax.set_facecolor("#1a1a2e")
    ok  = [(h["step"], h["lap_time"]) for h in LAP_HISTORY if h["lap_time"] is not None]
    dnf = [h["step"] for h in LAP_HISTORY if h["lap_time"] is None]
    if BASELINE["lap_time"]:
        ax.axhline(BASELINE["lap_time"], color=MPC_COLOR, ls="--", lw=1.5, label="MPC alone")
    if ok:
        xs, ys = zip(*ok)
        ax.plot(xs, ys, color=RL_COLOR, lw=2, marker="o", markersize=4, label="MPC + RL")
    if dnf:
        ax.plot(dnf, [max(ys) if ok else BASELINE["lap_time"] or 0] * len(dnf), "x",
                color="#ff1744", markersize=9, mew=2, label="DNF")
    ax.legend(fontsize=9, facecolor="#1a1a2e", labelcolor="white")
    ax.set_title("eval lap time (s) ↓", color="white", fontsize=11, pad=4)
    ax.set_xlabel("Decisions", color="#aaa", fontsize=9)
    ax.tick_params(colors="#aaa", labelsize=9)
    for spine in ax.spines.values(): spine.set_edgecolor("#333")

    step = _live["steps"][-1]
    wandb_url = wandb.run.url if wandb.run else "not connected"
    fig.suptitle(f"PPO from Scratch: Residual MPC  |  Step {step:,} / {total_steps:,} "
                 f"({step / total_steps * 100:.1f}%)   |   WandB: {wandb_url}",
                 color="white", fontsize=11, x=0.02, ha="left")
    clear_output(wait=True)
    plt.show()
    plt.close(fig)

print("Live plot helper defined.")

## 7.5 Train

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PPO from Scratch — Residual MPC Training (run this one cell)
# ══════════════════════════════════════════════════════════════════════════════
# Same PPO as Learn to Drive, with small starting exploration, lr/entropy
# annealing and a best-lap checkpoint. Every EVAL_FREQ decisions we time one
# lap and store it in LAP_HISTORY for the plots in Section 9.
N_STEPS       = 512         # decisions collected per env per update
N_EPOCHS      = 10
MINIBATCH     = 256
GAMMA         = 0.99
GAE_LAMBDA    = 0.95
CLIP_EPS      = 0.2
ENT_COEF      = 0.001       # starting entropy bonus (annealed to 0)
VF_COEF       = 0.5
LR            = 3e-4        # starting learning rate, annealed linearly to 0
MAX_GRAD      = 0.5
INIT_LOG_STD  = -1.0        # policy std starts at exp(-1) = 0.37: small corrections at first

torch.manual_seed(42)
np.random.seed(42)

run = wandb.init(
    project = "f1tenth-rl-lecture1",
    name    = f"ppo_scratch_residual_mpc_{MAP_NAME}_{TOTAL_STEPS//1000}k",
    config  = dict(n_steps=N_STEPS, n_epochs=N_EPOCHS, gamma=GAMMA, gae_lambda=GAE_LAMBDA,
                   clip_eps=CLIP_EPS, ent_coef=ENT_COEF, lr=LR, lr_schedule="linear",
                   map=MAP_NAME, num_envs=NUM_ENVS, res_steer_max=RES_STEER_MAX,
                   res_speed_max=RES_SPEED_MAX, v_max=V_MAX, crash_penalty=CRASH_PENALTY,
                   wall_prox_coef=WALL_PROX_COEF, residual_coef=RESIDUAL_COEF,
                   mpc_baseline_lap=BASELINE["lap_time"]),
)
wandb.define_metric("global_step")
wandb.define_metric("*", step_metric="global_step")
print(f"🔗 WandB: {run.url}\n")

train_env = make_training_env(NUM_ENVS)
eval_env  = ResidualMPCWrapper()           # timed laps from the start line
obs_dim   = train_env.observation_space.shape[0]
act_dim   = train_env.action_space.shape[0]

device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
net    = ActorCritic(obs_dim, act_dim).to(device)
net.log_std.data.fill_(INIT_LOG_STD)       # start close to the MPC
opt    = optim.Adam(net.parameters(), lr=LR, eps=1e-5)
print(net)
print(f"Parameters: {sum(p.numel() for p in net.parameters()):,}\n")

episode_count = 0
cur_rew  = np.zeros(NUM_ENVS)
cur_spd  = [[] for _ in range(NUM_ENVS)]
cur_col  = np.zeros(NUM_ENVS, dtype=bool)
cur_len  = np.zeros(NUM_ENVS, dtype=int)
ep_rew_window   = deque(maxlen=100)
ep_len_window   = deque(maxlen=100)
ep_crash_window = deque(maxlen=100)       # 1 = that episode ended in a crash

_live.update({k: [] for k in _live})
LAP_HISTORY.clear()
os.makedirs("models_residual_mpc", exist_ok=True)
BEST = {"lap_time": None, "step": None}


def eval_lap(step):
    """Time one deterministic lap, keep it for the plots, checkpoint the best, log to WandB."""
    lap = run_lap(eval_env, greedy_policy(net))
    LAP_HISTORY.append({"step": step, **lap})
    if lap["lap_time"] is not None and (BEST["lap_time"] is None or lap["lap_time"] < BEST["lap_time"]):
        BEST.update(lap_time=lap["lap_time"], step=step)
        torch.save(net.state_dict(), "models_residual_mpc/ppo_scratch_residual_mpc_best.pt")
    log = {"eval/lap_distance": lap["dist"], "eval/lap_completed": float(lap["lap_time"] is not None),
           "global_step": step}
    if lap["lap_time"] is not None:
        log["eval/lap_time"] = lap["lap_time"]
        log["eval/gain_vs_mpc"] = (BASELINE["lap_time"] or lap["lap_time"]) - lap["lap_time"]
    wandb.log(log)
    return lap


eval_lap(0)   # untrained policy, should be close to the MPC

obs = torch.tensor(train_env.reset(), dtype=torch.float32, device=device)
global_step    = 0
last_eval_step = 0
n_updates      = TOTAL_STEPS // (N_STEPS * NUM_ENVS)
print(f"Training {TOTAL_STEPS:,} decisions = {n_updates} updates; plots update below\n")

for update in range(n_updates):
    # anneal lr and entropy linearly to 0
    lr_now  = LR * (1.0 - update / n_updates)
    ent_now = ENT_COEF * (1.0 - update / n_updates)
    for g in opt.param_groups:
        g["lr"] = lr_now

    mb_obs, mb_act, mb_logp, mb_val, mb_rew, mb_done = [], [], [], [], [], []
    update_speeds, update_res = [], []

    for _ in range(N_STEPS):
        action, logp, value = net.act(obs)
        clipped_np = np.clip(action.cpu().numpy(), -1.0, 1.0)
        next_obs, reward, dones, infos = train_env.step(clipped_np)
        raw_reward = train_env.get_original_reward()
        update_res.append(float(np.mean(np.abs(clipped_np))))

        mb_obs.append(obs); mb_act.append(action); mb_logp.append(logp); mb_val.append(value)
        mb_rew.append(torch.tensor(reward, dtype=torch.float32, device=device))
        mb_done.append(torch.tensor(dones, dtype=torch.float32, device=device))

        cur_rew += raw_reward
        cur_len += 1
        for i, inf in enumerate(infos):
            cur_spd[i].append(inf.get("speed", 0.0))
            update_speeds.append(inf.get("speed", 0.0))
            if inf.get("collision", False) or inf.get("spun_out", False): cur_col[i] = True

        # bootstrap episodes that hit the time limit (not crashes)
        for i, d in enumerate(dones):
            if d and infos[i].get("TimeLimit.truncated", False):
                term_obs = torch.tensor(infos[i]["terminal_observation"], dtype=torch.float32,
                                        device=device).unsqueeze(0)
                with torch.no_grad():
                    _, term_value = net(term_obs)
                mb_rew[-1][i] += GAMMA * term_value.item()

        for i, d in enumerate(dones):
            if d:
                episode_count += 1
                ep_rew_window.append(cur_rew[i]); ep_len_window.append(cur_len[i])
                ep_crash_window.append(float(cur_col[i]))
                wandb.log({
                    "racing/avg_speed":          float(np.mean(cur_spd[i])) if cur_spd[i] else 0.0,
                    "racing/crash_rate_last100": float(np.mean(ep_crash_window)),
                    "racing/episode_count":      episode_count,
                    "rollout/ep_rew_mean":       float(np.mean(ep_rew_window)),
                    "rollout/ep_len_mean":       float(np.mean(ep_len_window)),
                    "global_step":               global_step,
                })
                cur_rew[i] = 0.0; cur_spd[i] = []; cur_col[i] = False; cur_len[i] = 0

        obs = torch.tensor(next_obs, dtype=torch.float32, device=device)
        global_step += NUM_ENVS

    with torch.no_grad():
        _, last_value = net(obs)
    values = torch.stack(mb_val)
    advantages, returns = compute_gae(torch.stack(mb_rew), values, torch.stack(mb_done),
                                      last_value, gamma=GAMMA, lam=GAE_LAMBDA)
    b_obs     = torch.stack(mb_obs).reshape(-1, obs_dim)
    b_act     = torch.stack(mb_act).reshape(-1, act_dim)
    b_logp    = torch.stack(mb_logp).reshape(-1)
    b_adv     = advantages.reshape(-1)
    b_ret     = returns.reshape(-1)
    b_val_old = values.reshape(-1)

    batch_size = b_obs.shape[0]
    idxs = np.arange(batch_size)
    epoch_losses, epoch_pg, epoch_vf = [], [], []
    for epoch in range(N_EPOCHS):
        np.random.shuffle(idxs)
        for start in range(0, batch_size, MINIBATCH):
            mb = idxs[start:start + MINIBATCH]
            dist, value = net(b_obs[mb])
            new_logp = dist.log_prob(b_act[mb]).sum(-1)
            entropy  = dist.entropy().sum(-1).mean()

            mb_adv = b_adv[mb]
            mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)

            ratio       = torch.exp(new_logp - b_logp[mb])
            surr1       = ratio * mb_adv
            surr2       = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * mb_adv
            policy_loss = -torch.min(surr1, surr2).mean()
            value_loss  = ((value - b_ret[mb]) ** 2).mean()
            loss        = policy_loss + VF_COEF * value_loss - ent_now * entropy

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), MAX_GRAD)
            opt.step()

            epoch_losses.append(loss.item())
            epoch_pg.append(policy_loss.item())
            epoch_vf.append(value_loss.item())

    y_true, y_pred = b_ret.cpu().numpy(), b_val_old.cpu().numpy()
    var_y = np.var(y_true)
    explained_var = float("nan") if var_y == 0 else 1.0 - np.var(y_true - y_pred) / var_y
    current_std   = net.log_std.exp().mean().item()
    wandb.log({
        "train/loss":                 float(np.mean(epoch_losses)),
        "train/policy_gradient_loss": float(np.mean(epoch_pg)),
        "train/value_loss":           float(np.mean(epoch_vf)),
        "train/std":                  current_std,
        "train/explained_variance":   explained_var,
        "train/learning_rate":        lr_now,
        "train/mean_residual":        float(np.mean(update_res)),
        "global_step":                global_step,
    })

    # timed eval lap
    if global_step - last_eval_step >= EVAL_FREQ or update == n_updates - 1:
        last_eval_step = global_step
        lap = eval_lap(global_step)
        print(f"  [eval lap at step {global_step:,}] {lap_str(lap)}  (MPC alone: {lap_str(BASELINE)})")

    if episode_count > 0:
        _live["steps"].append(global_step)
        _live["rew"].append(float(np.mean(ep_rew_window)))
        _live["crash"].append(float(np.mean(ep_crash_window)))
        _live["speed"].append(float(np.mean(update_speeds)) if update_speeds else 0.0)
        _live["loss"].append(float(np.mean(epoch_losses)))
        _live["residual"].append(float(np.mean(update_res)))
        draw_live_plots(TOTAL_STEPS)
        best_txt = f"{BEST['lap_time']:.2f} s" if BEST["lap_time"] else "none yet"
        print(f"Update {update+1}/{n_updates} | step {global_step:,} | episodes={episode_count} | "
              f"ep_rew_mean={np.mean(ep_rew_window):.1f} | crash rate={np.mean(ep_crash_window):.2f} | "
              f"lr={lr_now:.1e} | best lap={best_txt} (MPC alone {lap_str(BASELINE)})")

torch.save(net.state_dict(), "models_residual_mpc/ppo_scratch_residual_mpc.pt")
train_env.save("models_residual_mpc/vecnormalize.pkl")
np.save("models_residual_mpc/lap_history.npy", np.array(LAP_HISTORY, dtype=object), allow_pickle=True)
np.savetxt("models_residual_mpc/waypoints.csv", WAYPOINTS, delimiter=",")
print("\nSaved: models_residual_mpc/ppo_scratch_residual_mpc.pt       (final network)")
print(f"Saved: models_residual_mpc/ppo_scratch_residual_mpc_best.pt  (fastest eval lap: "
      f"{BEST['lap_time']:.2f} s at step {BEST['step']:,})" if BEST["lap_time"] else
      "No eval lap finished, so no best checkpoint was saved.")
print("Saved: models_residual_mpc/vecnormalize.pkl")
print("Saved: models_residual_mpc/lap_history.npy                  (every eval lap, for the plots)")
print("Saved: models_residual_mpc/waypoints.csv                    (the MPC's waypoints)")
wandb.finish()
train_env.close()

---

# **Section 8 — Save Your Trained Model** 💾

The archive holds the best and final policy weights, the reward-normalization stats, every
eval lap (`lap_history.npy`) and the MPC's `waypoints.csv`, so the residual can be replayed on
top of the same MPC later.

In [ ]:
!zip -r models_residual_mpc_archive.zip /content/models_residual_mpc

In [ ]:
from google.colab import files
files.download('models_residual_mpc_archive.zip')

---

# **Section 9 — Evaluation & Visualization** 🎬

## 9.1 MPC Alone vs MPC + RL

Load the **best-lap checkpoint**, time it from the start line and compare it with the MPC on
its own. To make sure it's not a one-lap fluke, we also do **10 runs of 30 s from random
points on the track** with both controllers and count the crashes.

In [ ]:
# ── MPC alone vs MPC + RL, plus robustness ────────────────────────────────────
ckpt = ("models_residual_mpc/ppo_scratch_residual_mpc_best.pt"
        if os.path.exists("models_residual_mpc/ppo_scratch_residual_mpc_best.pt")
        else "models_residual_mpc/ppo_scratch_residual_mpc.pt")
eval_net = ActorCritic(obs_dim, act_dim).to("cpu")
eval_net.load_state_dict(torch.load(ckpt, map_location="cpu"))
eval_net.eval()
print(f"Loaded {ckpt}\n")

FINAL = run_lap(eval_env, greedy_policy(eval_net))
RESULTS = {"MPC alone": BASELINE, "MPC + RL residual (ours)": FINAL}

print("─" * 70)
print(f"  {'Controller':<28} {'Lap time':>16} {'Avg speed':>10} {'Top speed':>10}")
print("─" * 70)
for name, r in RESULTS.items():
    print(f"  {name:<28} {lap_str(r):>16} {np.mean(r['path'][:, 2]):>6.2f} m/s {np.max(r['path'][:, 2]):>6.2f} m/s")
print("─" * 70)
if BASELINE["lap_time"] and FINAL["lap_time"]:
    gain = BASELINE["lap_time"] - FINAL["lap_time"]
    print(f"  The residual saves {gain:.2f} s per lap ({gain / BASELINE['lap_time'] * 100:.1f}% faster)")


def robustness(policy_fn, n=10):
    crashes, speeds = 0, []
    for k in range(n):
        obs, _ = eval_env.reset(seed=1000 + k)
        done = trunc = False
        while not (done or trunc):
            obs, _, done, trunc, info = eval_env.step(policy_fn(obs))
            speeds.append(info["speed"])
        crashes += int(info["collision"] or info["spun_out"])
    return n - crashes, float(np.mean(speeds))

for name, pi in [("MPC alone", zero_residual), ("MPC + RL residual", greedy_policy(eval_net))]:
    ok, spd = robustness(pi)
    print(f"  Robustness, {name:<18}: {ok}/10 random-start runs of 30 s without a crash "
          f"(average speed {spd:.2f} m/s)")

names = [n for n, r in RESULTS.items() if r["lap_time"]]
times = [RESULTS[n]["lap_time"] for n in names]
fig, ax = plt.subplots(figsize=(10, 3))
bars = ax.barh(names, times, color=[MPC_COLOR, RL_COLOR][:len(names)])
ax.bar_label(bars, fmt="%.2f s", padding=4)
ax.set_xlabel("lap time (s), lower is better"); ax.invert_yaxis()
ax.set_xlim(0, max(times) * 1.15); ax.set_title("Lap time comparison")
plt.tight_layout(); plt.show()

## 9.2 Where Does the Residual Act?

One nice thing about a residual is that we can see exactly where, and by how much, the policy
changes the MPC's command. Three plots, all against distance along the centerline:
- **Speed** of the MPC alone and of MPC + RL
- **Speed correction** $\Delta v$ the policy adds (positive = faster than the MPC asked)
- **Steering correction** $\Delta\delta$ the policy adds

The red bands are the sharpest corners C1 to C3.

In [ ]:
# ── Residual commands along the lap ───────────────────────────────────────────
_cl_t = np.column_stack([np.cos(CL.yaws), np.sin(CL.yaws)])      # centerline tangents


def along_track(path):
    """For each path point: distance along the centerline and signed lateral offset."""
    xy = path[:, :2]
    d2 = ((xy[:, None, :] - CL_XY[None, ::2, :]) ** 2).sum(-1)    # every 2nd centerline point
    j  = np.argmin(d2, axis=1) * 2
    rel = xy - CL_XY[j]
    lat = _cl_t[j, 0] * rel[:, 1] - _cl_t[j, 1] * rel[:, 0]        # + is left of the centerline
    return np.asarray(CL.ss)[j], lat


fig, (ax_v, ax_dv, ax_dd) = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for s0 in CORNER_S:
    for ax in (ax_v, ax_dv, ax_dd):
        ax.axvspan(s0 - 6, s0 + 6, color="red", alpha=0.08)
for name, r, c in [("MPC alone", BASELINE, MPC_COLOR), ("MPC + RL", FINAL, RL_COLOR)]:
    if len(r["path"]) > 1:
        s, _ = along_track(r["path"]); o = np.argsort(s)
        ax_v.plot(s[o], r["path"][o, 2], color=c, lw=1.6, label=name)
if len(FINAL["residuals"]):
    dec_path = FINAL["path"][SIM_STEPS - 1::SIM_STEPS][:len(FINAL["residuals"])]   # car position at each decision
    s, _ = along_track(dec_path); o = np.argsort(s)
    res = FINAL["residuals"][:len(dec_path)]
    ax_dv.plot(s[o], res[o, 1], color="#43a047", lw=1.4)
    ax_dd.plot(s[o], res[o, 0], color="#8e24aa", lw=1.4)
for ax in (ax_dv, ax_dd):
    ax.axhline(0, color="k", lw=0.8)
ax_v.set_ylabel("speed (m/s)"); ax_v.legend(); ax_v.set_title("Speed: MPC alone vs MPC + RL")
ax_dv.set_ylabel("Δspeed (m/s)"); ax_dv.set_title("Speed correction added by the policy")
ax_dd.set_ylabel("Δsteer (rad)"); ax_dd.set_title("Steering correction added by the policy")
ax_dd.set_xlabel("distance along the centerline (m)")
for ax in (ax_v, ax_dv, ax_dd): ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9.3 The Two Lines, Corner by Corner

The full lap and the three sharpest corners, with the MPC-alone path in cyan and the
MPC + RL path coloured by its speed.

In [ ]:
# ── Path comparison: full lap + 3 corners ─────────────────────────────────────
fig, axs = plt.subplots(1, 4, figsize=(22, 6), gridspec_kw={"width_ratios": [1.6, 1, 1, 1]})
track_ax(axs[0])
plain_line(axs[0], BASELINE["path"][:, :2], MPC_COLOR, "MPC alone", lw=1.5)
lc = speed_line(axs[0], FINAL["path"], lw=2.0)
axs[0].set_title(f"Full lap: MPC {lap_str(BASELINE)} vs MPC + RL {lap_str(FINAL)}"); axs[0].legend(loc="lower left")
for k, ax in enumerate(axs[1:]):
    track_ax(ax, CORNER_XY[k], zoom=8)
    plain_line(ax, CL_XY, CL_COLOR, "centerline (waypoints)", ls="--", lw=1.2)
    plain_line(ax, BASELINE["path"][:, :2], MPC_COLOR, "MPC alone", lw=2)
    speed_line(ax, FINAL["path"], lw=3.5)
    ax.set_title(f"Corner C{k+1}")
axs[1].legend(loc="lower left", fontsize=9)
fig.colorbar(lc, ax=axs, fraction=0.012, label="MPC + RL speed (m/s)")
plt.show()

## 9.4 How the Residual Evolved

Eval laps from different points in training. The first is basically the MPC with a tiny
untrained correction; watch the speed (colour) go up where the policy learned the MPC was too
cautious.

In [ ]:
# ── Path evolution: 4 snapshots across training ───────────────────────────────
pool  = [h for h in LAP_HISTORY if len(h["path"]) > 1]
picks = [pool[0], pool[len(pool) // 3], pool[2 * len(pool) // 3], pool[-1]] if len(pool) >= 4 else pool

fig, axs = plt.subplots(1, len(picks), figsize=(5 * len(picks), 4.5), squeeze=False)
for k, h in enumerate(picks):
    track_ax(axs[0, k]); lc = speed_line(axs[0, k], h["path"], lw=2.0)
    axs[0, k].set_title(f"step {h['step']:,}\nlap {lap_str(h)}")
fig.colorbar(lc, ax=axs, fraction=0.015, label="speed (m/s)")
plt.show()

## 9.5 Is It Physically Sensible?

Sideways acceleration along the lap, measured from the simulator state as speed times yaw
rate (the same filtered signal the spin-out rule uses). A good residual uses **more of the
grip than the cautious MPC** but never goes past the tyres' real limit.

In [ ]:
# ── Lateral acceleration: MPC alone vs MPC + RL ───────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4.5))
for name, r, c in [("MPC alone", BASELINE, MPC_COLOR), ("MPC + RL", FINAL, RL_COLOR)]:
    if len(r["path"]) < 50:
        continue
    p = r["path"][::5]
    s, _ = along_track(p); o = np.argsort(s)
    ax.plot(s[o], p[o, 3], color=c, lw=1.5, label=f"{name} (peak {np.max(p[:, 3]):.1f} m/s²)")
ax.axhline(MU_G, color="red", ls="--", lw=1.5, label=f"real tyre grip limit, about {MU_G:.1f} m/s²")
ax.axhline(SPIN_OUT_LIMIT, color="darkred", ls="-", lw=1, label=f"spin-out, {SPIN_OUT_LIMIT:.1f} m/s²")
for s0 in CORNER_S:
    ax.axvspan(s0 - 6, s0 + 6, color="red", alpha=0.08)
ax.set_xlabel("distance along the centerline (m)"); ax.set_ylabel("lateral acceleration (m/s²)")
ax.set_title("How much grip each controller uses (red bands are corners C1 to C3)")
ax.set_ylim(0, SPIN_OUT_LIMIT * 1.2); ax.legend(loc="upper right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9.6 Record Eval Video

In [ ]:
# ── Record a video of MPC + RL in the simulator ──────────────────────────────
import imageio

vid_env = ResidualMPCWrapper(render_mode="rgb_array")
obs, _ = vid_env.reset(options={"poses": START_POSE})
policy = greedy_policy(eval_net)
frames, dist, n = [], 0.0, 0
while dist < vid_env.track_len and n < 2000:
    obs, _, done, _, info = vid_env.step(policy(obs))
    dist += info["ds"]; n += 1
    frame = vid_env.render()   # one frame per decision
    if frame is not None:
        frames.append(frame)
    if info["collision"] or info["spun_out"]:
        print("Crashed during the video lap"); break
vid_env.close()

if frames:
    imageio.mimsave("eval_render_residual_mpc.mp4", frames, fps=10)
    print(f"✅ Captured {len(frames)} frames ({n * SIM_STEPS * vid_env.dt:.1f} s of driving)")
    from google.colab import files
    files.download("eval_render_residual_mpc.mp4")
else:
    print("❌ No frames captured")

---

# **Section 10 — Summary** 📝

## What We Built

1. **A residual on top of a real controller:** the Lab 8 MPC drives, and the policy adds a
   small correction $u = u_{\text{MPC}} + u_{\text{RL}}$ to its steering and speed.
2. **Safe learning from step one:** the untrained policy *is* the MPC (mean correction zero,
   small initial exploration), so training never starts from a crashing car.
3. **The same reward terms as before:** progress, the wall penalty and the crash penalty, plus
   one new term, the residual penalty, so the policy only overrides the MPC where it pays off.
4. **Real-car physics:** the same spin-out rule as Learn the Racing Line, so the corrections
   never ask the tyres for more than a real car could give.
5. **An interpretable result:** we can see exactly where the policy overrides the MPC, and by
   how much, along the lap.

The PPO (network, GAE, clipped loss) is the same one we built in Learn to Drive. What's
different this time is that we didn't replace the classical controller; we **built on top of it**.

## Things to Try

| Experiment | What to look for |
|---|---|
| Raise `RES_SPEED_MAX` to 3 m/s | Does the lap get faster, or do crashes go up? |
| Set `RESIDUAL_COEF = 0` | Do the corrections get larger and noisier? |
| A faster waypoint profile (`PROFILE_A_LAT = 6`) | Does a better MPC leave less for the residual to gain? |
| Detune the MPC (`Qk` ten times smaller) | Can the residual rescue a badly tuned controller? |
| Add lidar noise or a friction change | Does the residual still help? This is the sim-to-real question |